# Online Retail Sales Performance Analysis

**Business question:** How is revenue trending over time, which products and regions drive the most value, and where are the opportunities to grow average order value?

**Dataset:** Online Retail II — a UK-based online gift-ware retailer, Dec 2009–Dec 2011. This notebook works from an already-cleaned version of the raw dataset (see `cleaning_script.py` in this repo for the cleaning logic and decisions).

**Tools:** SQL (via DuckDB) for aggregation, Python/pandas for derived metrics and validation, Tableau Public for the final interactive dashboard.

## Contents
1. [Monthly Revenue Trend](#1.-Monthly-Revenue-Trend)
2. [Top Products: Revenue vs. Quantity](#2.-Top-Products:-Revenue-vs.-Quantity)
3. [Revenue by Country](#3.-Revenue-by-Country)
4. [Average Order Value (AOV)](#4.-Average-Order-Value-(AOV))
5. [Analysis Summary](#Analysis-Summary)


In [1]:
import pandas as pd
import duckdb

In [2]:
data = pd.read_csv("cleaned_retail.csv")
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])

con = duckdb.connect()
con.register("retail", data)

print(data.shape)

(804824, 9)


## 1. Monthly Revenue Trend

In [3]:
monthly_revenue = con.sql("""
SELECT
    DATE_TRUNC('month', InvoiceDate) AS Month,
    SUM(Quantity * Price) AS Revenue
FROM retail
GROUP BY Month
ORDER BY Month
""").df()

print(monthly_revenue)

        Month      Revenue
0  2009-12-01   684320.220
1  2010-01-01   538069.372
2  2010-02-01   502669.686
3  2010-03-01   672300.791
4  2010-04-01   591821.772
5  2010-05-01   598287.540
6  2010-06-01   636251.100
7  2010-07-01   586884.030
8  2010-08-01   599898.100
9  2010-09-01   812929.061
10 2010-10-01  1022593.490
11 2010-11-01  1171267.312
12 2010-12-01   884159.470
13 2011-01-01   569335.640
14 2011-02-01   446712.020
15 2011-03-01   590587.850
16 2011-04-01   459076.861
17 2011-05-01   674276.650
18 2011-06-01   660442.580
19 2011-07-01   598543.241
20 2011-08-01   642354.360
21 2011-09-01   949377.141
22 2011-10-01  1017745.160
23 2011-11-01  1158801.060
24 2011-12-01   517757.690


### Data Quality Note: Incomplete Final Month

The dataset's transaction records end on **2011-12-09**, meaning December 2011 contains only 9 days of data compared to a full month (30 days) for every other period in the dataset.

**Impact:** December 2011 shows $517,757 in revenue — notably lower than November 2011 ($1,158,801) and December 2010 ($884,159). Without this context, this could be misread as a real sales decline.

**Investigation:** Checked the dataset's actual date range (established during data cleaning) and confirmed the December 2011 figure reflects only 9 days of transactions, not a full month.

**Decision:** December 2011 is excluded from month-over-month trend analysis and visualizations to avoid a misleading comparison. This is noted explicitly on the dashboard rather than silently dropping the data point.

In [4]:
monthly_revenue_clean = monthly_revenue[monthly_revenue["Month"] < "2011-12-01"].copy()

monthly_revenue_clean["Revenue_Growth_Pct"] = (
    monthly_revenue_clean["Revenue"].pct_change() * 100
)

print(monthly_revenue_clean)

        Month      Revenue  Revenue_Growth_Pct
0  2009-12-01   684320.220                 NaN
1  2010-01-01   538069.372          -21.371698
2  2010-02-01   502669.686           -6.579019
3  2010-03-01   672300.791           33.746038
4  2010-04-01   591821.772          -11.970686
5  2010-05-01   598287.540            1.092519
6  2010-06-01   636251.100            6.345370
7  2010-07-01   586884.030           -7.759055
8  2010-08-01   599898.100            2.217486
9  2010-09-01   812929.061           35.511191
10 2010-10-01  1022593.490           25.791233
11 2010-11-01  1171267.312           14.538898
12 2010-12-01   884159.470          -24.512580
13 2011-01-01   569335.640          -35.607132
14 2011-02-01   446712.020          -21.538019
15 2011-03-01   590587.850           32.207736
16 2011-04-01   459076.861          -22.267811
17 2011-05-01   674276.650           46.876636
18 2011-06-01   660442.580           -2.051691
19 2011-07-01   598543.241           -9.372403
20 2011-08-01

### Finding: Consistent Seasonal Pattern

Revenue growth shows a strong, repeating seasonal pattern across both years in the dataset:
- **September**: sharp growth spike (+35.5% in 2010, +47.8% in 2011)
- **October–November**: continued strong growth, peaking ahead of the holidays
- **January**: sharp post-holiday decline (-21.4% in 2010, -35.6% in 2011)

**Why this matters:** the fact that this pattern repeats independently in both 2010 and 2011 — same calendar months, same direction, similar magnitude — rules out a one-time cause (e.g., a single marketing campaign or data anomaly) and strongly supports a genuine, recurring seasonal demand cycle tied to holiday gift shopping. A single year's pattern alone would be an anecdote; two consecutive years showing the same shape is real evidence.

**Business implication:** inventory and marketing planning should anticipate this cycle — building stock ahead of September, and expecting a natural demand drop in January rather than treating it as a concerning decline.

## 2. Top Products: Revenue vs. Quantity

In [5]:
top_products_revenue = con.sql("""
SELECT
    StockCode,
    Description,
    SUM(Quantity * Price) AS Revenue,
    SUM(Quantity) AS Total_Quantity
FROM retail
GROUP BY StockCode, Description
ORDER BY Revenue DESC
LIMIT 10
""").df()

print(top_products_revenue)

  StockCode                         Description    Revenue  Total_Quantity
0     22423            REGENCY CAKESTAND 3 TIER  286486.30         24914.0
1    85123A  WHITE HANGING HEART T-LIGHT HOLDER  252072.46         93640.0
2     23843         PAPER CRAFT , LITTLE BIRDIE  168469.60         80995.0
3    85099B             JUMBO BAG RED RETROSPOT  136980.08         75759.0
4     84879       ASSORTED COLOUR BIRD ORNAMENT  127074.17         79913.0
5      POST                             POSTAGE  126563.04          5333.0
6     47566                       PARTY BUNTING  103880.23         23611.0
7     23166      MEDIUM CERAMIC TOP STORAGE JAR   81416.73         77916.0
8     22086     PAPER CHAIN KIT 50'S CHRISTMAS    79594.33         29477.0
9     79321                       CHILLI LIGHTS   72860.14         15735.0


**Why exclude shipping from this ranking?** Monetary value at the customer level (used in the companion segmentation project) correctly includes postage, since it's real revenue the business collected. But "top products by revenue" is answering a different question — which physical products should the business prioritize stocking, marketing, or negotiating better supplier terms on. Postage isn't a merchandising decision; it's a pass-through fee tied to order volume. Including it here would misleadingly suggest postage is a top-performing product line. **Decision:** exclude non-product codes (`POST`, `DOT`, `C2`) from product-level rankings specifically.

In [6]:
top_products_revenue = con.sql("""
SELECT
    StockCode,
    Description,
    SUM(Quantity * Price) AS Revenue,
    SUM(Quantity) AS Total_Quantity
FROM retail
WHERE StockCode NOT IN ('POST', 'DOT', 'C2')
GROUP BY StockCode, Description
ORDER BY Revenue DESC
LIMIT 10
""").df()

print(top_products_revenue)

  StockCode                         Description    Revenue  Total_Quantity
0     22423            REGENCY CAKESTAND 3 TIER  286486.30         24914.0
1    85123A  WHITE HANGING HEART T-LIGHT HOLDER  252072.46         93640.0
2     23843         PAPER CRAFT , LITTLE BIRDIE  168469.60         80995.0
3    85099B             JUMBO BAG RED RETROSPOT  136980.08         75759.0
4     84879       ASSORTED COLOUR BIRD ORNAMENT  127074.17         79913.0
5     47566                       PARTY BUNTING  103880.23         23611.0
6     23166      MEDIUM CERAMIC TOP STORAGE JAR   81416.73         77916.0
7     22086     PAPER CHAIN KIT 50'S CHRISTMAS    79594.33         29477.0
8     79321                       CHILLI LIGHTS   72860.14         15735.0
9     21137            BLACK RECORD COVER FRAME   67209.44         19629.0


In [7]:
top_products_quantity = con.sql("""
SELECT
    StockCode,
    Description,
    SUM(Quantity) AS Total_Quantity,
    SUM(Quantity * Price) AS Revenue
FROM retail
WHERE StockCode NOT IN ('POST', 'DOT', 'C2')
GROUP BY StockCode, Description
ORDER BY Total_Quantity DESC
LIMIT 10
""").df()

print(top_products_quantity)

  StockCode                         Description  Total_Quantity    Revenue
0     84077   WORLD WAR 2 GLIDERS ASSTD DESIGNS        109169.0   24905.87
1    85123A  WHITE HANGING HEART T-LIGHT HOLDER         93640.0  252072.46
2     23843         PAPER CRAFT , LITTLE BIRDIE         80995.0  168469.60
3     84879       ASSORTED COLOUR BIRD ORNAMENT         79913.0  127074.17
4     23166      MEDIUM CERAMIC TOP STORAGE JAR         77916.0   81416.73
5    85099B             JUMBO BAG RED RETROSPOT         75759.0  136980.08
6     17003                 BROCADE RING PURSE          71129.0   14827.71
7     21977  PACK OF 60 PINK PAISLEY CAKE CASES         55270.0   26733.45
8     84991         60 TEATIME FAIRY CAKE CASES         53495.0   26121.57
9     21212     PACK OF 72 RETROSPOT CAKE CASES         46107.0   22214.26


### Finding: Revenue vs. Volume Tell Different Stories

Comparing "top products by revenue" against "top products by quantity sold" reveals two distinct sets of high-performing products, not one.

**Example:** WORLD WAR 2 GLIDERS (StockCode 84077) sold 109,169 units — the highest quantity of any product — but generated only $24,905.87 in revenue (~$0.23/unit). By contrast, REGENCY CAKESTAND 3 TIER sold far fewer units (24,914) but generated $286,486.30 (~$11.50/unit) — the highest revenue of any single product.

**Business implication:** revenue and volume leaders serve different purposes. High-revenue items (cakestands, bunting) are margin drivers worth prioritizing in marketing and merchandising. High-volume, low-price items (gliders, cake cases) likely drive basket size, impulse purchases, or repeat orders even though each unit contributes little revenue individually. Reporting only one ranking would miss half the picture.

## 3. Revenue by Country

In [8]:
country_revenue = con.sql("""
SELECT
    Country,
    SUM(Quantity * Price) AS Revenue,
    COUNT(DISTINCT "Customer ID") AS Num_Customers
FROM retail
GROUP BY Country
ORDER BY Revenue DESC
""").df()

country_revenue["Pct_of_Total"] = (
    country_revenue["Revenue"] / country_revenue["Revenue"].sum()
) * 100

print(country_revenue.head(15))

            Country       Revenue  Num_Customers  Pct_of_Total
0    United Kingdom  1.464693e+07           5335     83.285267
1              EIRE  6.006616e+05              3      3.415477
2       Netherlands  5.540357e+05             22      3.150353
3           Germany  4.280610e+05            107      2.434037
4            France  3.406877e+05             94      1.937215
5         Australia  1.688347e+05             15      0.960026
6             Spain  1.075398e+05             38      0.611492
7       Switzerland  1.003653e+05             22      0.570697
8            Sweden  8.977004e+04             19      0.510450
9           Denmark  6.986219e+04             12      0.397250
10          Belgium  6.424477e+04             29      0.365308
11         Portugal  5.294364e+04             23      0.301048
12            Japan  4.713839e+04             10      0.268038
13           Norway  4.535298e+04             12      0.257886
14  Channel Islands  4.470376e+04             13      0

### Finding: UK's Revenue Share (83.3%) vs. Transaction Share (92%)

While the UK accounts for 92% of raw transaction rows, it represents a lower 83.3% of total revenue. This gap is explained by a small number of high-value non-UK accounts — most notably EIRE (Ireland), where just 3 customers generated $600,661 in total revenue, averaging $200,220 per customer versus a dataset-wide median of under $900.

This is consistent with the wholesale/B2B account pattern identified in the companion customer segmentation project — a handful of large business buyers can disproportionately skew country-level revenue figures, the same way they skew individual customer revenue totals. Country-level revenue comparisons should be read with this in mind, rather than assuming they reflect typical per-customer spending.

## 4. Average Order Value (AOV)

In [9]:
data["Revenue"] = data["Quantity"] * data["Price"]

invoice_revenue = (
    data.groupby("Invoice")
    .agg(
        Revenue=("Revenue", "sum"),
        Month=("InvoiceDate", "first"),
        Country=("Country", "first"),
    )
    .reset_index()
)

overall_aov = invoice_revenue["Revenue"].mean()
print("Overall Average Order Value:", round(overall_aov, 2))

Overall Average Order Value: 478.96


In [10]:
aov_by_month = invoice_revenue.groupby(invoice_revenue["Month"].dt.to_period("M"))[
    "Revenue"
].mean()
print(aov_by_month)

Month
2009-12    455.909540
2010-01    558.163249
2010-02    458.222139
2010-03    444.643380
2010-04    449.371125
2010-05    437.344693
2010-06    427.588105
2010-07    429.008794
2010-08    468.304528
2010-09    487.952618
2010-10    483.267245
2010-11    453.979578
2010-12    632.898690
2011-01    578.593130
2011-02    449.408471
2011-03    449.116236
2011-04    402.346066
2011-05    435.860795
2011-06    474.455876
2011-07    451.730748
2011-08    502.624695
2011-09    542.811401
2011-10    532.293494
2011-11    437.448494
2011-12    666.354813
Freq: M, Name: Revenue, dtype: float64


### Finding: December Shows Elevated Average Order Value

Both Decembers in the dataset show a notable AOV spike relative to surrounding months: December 2010 ($632.90) and December 2011 ($666.35), compared to a typical range of roughly $430–540 in other months.

This is a different pattern than the total-revenue finding for December 2011, where the incomplete month (only 9 days of data) showed a lower total. Average order value is unaffected by that incompleteness — it measures per-order spend, not total volume. The elevated AOV in both Decembers suggests customers place fewer but larger orders during this period, consistent with last-minute holiday shopping behavior (bundling multiple gift purchases into a single order) rather than spreading purchases across the month as usual.

**Business implication:** December marketing and fulfillment planning should anticipate larger basket sizes even if daily order counts are lower, rather than assuming reduced customer engagement.

### AOV by Country

In [11]:
aov_by_country = (
    invoice_revenue.groupby("Country")["Revenue"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
)
aov_by_country.columns = ["Avg_Order_Value", "Num_Orders"]
print(aov_by_country.head(15))

             Avg_Order_Value  Num_Orders
Country                                 
Netherlands      2473.373482         224
Australia        1875.940667          90
Lebanon          1693.880000           1
Singapore        1644.770000           8
Denmark          1624.702093          43
Thailand         1535.270000           2
Israel           1488.727143           7
Japan            1428.436061          33
EIRE             1131.189548         531
Switzerland      1115.170444          90
Lithuania        1092.290000           6
Norway           1079.832857          42
Greece           1060.899444          18
RSA               966.870000           2
Sweden            880.098431         102


### Finding: Small Sample Sizes Distort Country-Level AOV

Several countries show high average order values that are based on very few orders — Lebanon ($1,693.88 from just 1 order), Thailand ($1,535.27 from 2 orders), and RSA ($966.87 from 2 orders). These figures are not statistically meaningful and should not be interpreted as representative country-level spending patterns.

Countries with a reasonable order volume (e.g., Netherlands at 224 orders, EIRE at 531 orders, Switzerland at 90 orders) provide more reliable AOV comparisons, though EIRE's figure is still likely influenced by the small number of high-value wholesale accounts identified earlier in this analysis.

**Decision:** for the dashboard, country-level AOV is only displayed for countries with a minimum order threshold (20+ orders), with a note explaining the exclusion, rather than presenting all countries' figures as equally reliable.

In [12]:
aov_by_country_filtered = aov_by_country[aov_by_country["Num_Orders"] >= 20]
print(aov_by_country_filtered)

                 Avg_Order_Value  Num_Orders
Country                                     
Netherlands          2473.373482         224
Australia            1875.940667          90
Denmark              1624.702093          43
Japan                1428.436061          33
EIRE                 1131.189548         531
Switzerland          1115.170444          90
Norway               1079.832857          42
Sweden                880.098431         102
Channel Islands       827.847407          54
Spain                 721.743557         149
Cyprus                684.984444          36
Portugal              615.623721          86
France                562.190875         606
Germany               550.914995         777
Austria               543.093256          43
Finland               532.929091          55
Italy                 507.980469          64
United Kingdom        438.714788       33386
Belgium               434.086284         148
USA                   418.300500          20
Poland    

In [13]:
monthly_revenue_clean.to_csv("monthly_revenue.csv", index=False)
top_products_revenue.to_csv("top_products_revenue.csv", index=False)
top_products_quantity.to_csv("top_products_quantity.csv", index=False)
country_revenue.to_csv("country_revenue.csv", index=False)
aov_by_country_filtered.to_csv("aov_by_country.csv")

print("All tables exported.")

All tables exported.


## Analysis Summary

**1. Revenue by Month (Trend Analysis)**

Computed monthly revenue via SQL, month-over-month growth via pandas. Found a strong, repeating seasonal pattern across both years: sharp growth in September–November (holiday buildup), sharp decline in January (post-holiday slump). Identified December 2011 as an incomplete month (data ends Dec 9) and excluded it from trend analysis to avoid a misleading comparison.

**2. Top Products (Revenue vs. Quantity)**

Compared top 10 products by revenue against top 10 by quantity sold. Found these are largely different products — e.g., WORLD WAR 2 GLIDERS sold the most units (109,169) but generated modest revenue (~$0.23/unit), while REGENCY CAKESTAND 3 TIER generated the most revenue (~$11.50/unit) on far fewer units. Excluded non-product codes (POST, DOT, C2) from this ranking, since shipping isn't a merchandising decision.

**3. Revenue by Country**

UK represents 92% of transaction rows but only 83.3% of revenue. Traced the gap to a small number of high-value non-UK accounts (e.g., EIRE: 3 customers averaging $200,220 each) — consistent with the wholesale/B2B account pattern first identified in the companion segmentation project.

**4. Average Order Value (AOV)**

Calculated overall, by month, and by country. Found both Decembers show elevated AOV despite lower order counts — consistent with last-minute holiday shoppers placing fewer, larger orders. Found several countries' AOV figures were based on too few orders to be meaningful (e.g., Lebanon: $1,693 from a single order) and filtered the country-level AOV table to a 20+ order minimum before further use.

### Key business themes across all four analyses

Strong, predictable seasonality tied to holiday shopping. A small number of wholesale/B2B accounts distort multiple metrics (customer revenue totals, country revenue, country AOV) — a recurring pattern worth flagging to any stakeholder using this data. Revenue and volume are not interchangeable signals — both are needed for a complete picture of product performance.